# 01 - Data Download and Cleaning

This notebook downloads public international soccer match data, inspects columns, cleans the basic fields, creates a match outcome target, and saves cleaned data for later notebooks.

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [20]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR

(PosixPath('/Users/timothymcintosh/DSProjects/World-Cup-Predictor/data/raw'),
 PosixPath('/Users/timothymcintosh/DSProjects/World-Cup-Predictor/data/processed'))

In [21]:
RAW_DIR = "../data/raw"

In [23]:
results = pd.read_csv(f"{RAW_DIR}/results.csv")
shootouts = pd.read_csv(f"{RAW_DIR}/shootouts.csv")
world_cup_matches = pd.read_csv(f"{RAW_DIR}/matches.csv")

print(results.shape)
print(shootouts.shape)
print(world_cup_matches.shape)

(49493, 9)
(678, 5)
(1248, 37)


In [26]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [27]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 49493 entries, 0 to 49492
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49493 non-null  str    
 1   home_team   49493 non-null  str    
 2   away_team   49493 non-null  str    
 3   home_score  49478 non-null  float64
 4   away_score  49478 non-null  float64
 5   tournament  49493 non-null  str    
 6   city        49493 non-null  str    
 7   country     49493 non-null  str    
 8   neutral     49493 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [28]:
results.isna().sum()

date           0
home_team      0
away_team      0
home_score    15
away_score    15
tournament     0
city           0
country        0
neutral        0
dtype: int64

## Data Inspection

Before cleaning the dataset, we inspect its structure to understand the available variables, identify missing values, and determine what preprocessing steps to take.

### Key Observations
- The dataset contains one row per international men's soccer match.
- The 'date' column is initially stored as a string and will be converted to a datetime object.
- Only 'home_score' and 'away_score' contain missing values (15 rows each).
- All other columns are complete and used for features later in the project.

In [29]:
results["date"] = pd.to_datetime(results["date"])

In [ ]:
cutoff_date = pd.Timestamp("2026-06-10")

results = results[results["date"] <= cutoff_date]

## Filtering the Training Data

The model needs complete match dates, teams, scores, tournament labels, locations, and neutral-site indicators. Rows missing required values are removed because the match outcome cannot be calculated without complete score information.

In [ ]:
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VISUALS_DIR = PROJECT_ROOT / "visuals"
MODELS_DIR = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VISUALS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
results["date"].isna().sum()

In [ ]:
required_cols = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "city",
    "country",
    "neutral"
]

rows_before = len(results)
results = results.dropna(subset=required_cols).copy()
rows_after = len(results)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)

In [38]:
results["home_score"] = pd.to_numeric(results["home_score"])
results["away_score"] = pd.to_numeric(results["away_score"])

results["home_score"] = results["home_score"].astype(int)
results["away_score"] = results["away_score"].astype(int)

In [ ]:
results["neutral"] = (
    results["neutral"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False})
)

results["neutral"].value_counts(dropna=False)

In [ ]:
results["total_goals"] = (results["home_score"] + results["away_score"])

## Creating the Prediction Target

The target variable is the outcome the model will learn to predict. This project predicts three classes: `home_win`, `draw`, and `away_win`. The target is based only on the final score columns.


In [36]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["home_score"] < row["away_score"]:
        return "away_win"
    else:
        return "draw"

results["result"] = results.apply(get_result, axis=1)

In [ ]:
results["result"].value_counts(normalize=True)

In [ ]:
results = results.sort_values("date").reset_index(drop=True)

In [ ]:
clean_path = PROCESSED_DIR / "clean_matches.csv"

results.to_csv(clean_path, index=False)

clean_path

## Cleaning Summary

Cleaning steps completed:

- Converted `date` to datetime.
- Applied a cutoff date before the 2026 FIFA World Cup to avoid data leakage.
- Removed rows missing required fields.
- Converted score columns to integers.
- Standardized the neutral venue indicator.
- Created `total_goals` for analysis.
- Created the target variable with three classes: `home_win`, `draw`, and `away_win`.
- Sorted matches chronologically to prepare for rolling team-form features.

The cleaned dataset was saved as `data/processed/clean_matches.csv`.
